In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import time
from itertools import combinations
from pathlib import Path

%run 01_algorithms.ipynb

In [ ]:
def test_algorithm(algorithm, files, is_exact=False, max_nodes=30):
    for filepath in files:
        G, terminals, name = parse_stp(filepath)
        print(f'--- {name} ---')
        print(f'Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}, Terminals: {len(terminals)}')

        if is_exact and G.number_of_nodes() > max_nodes:
            print('Skipped (too many nodes for exact algorithm)')
            print()
            continue

        start = time.time()
        if is_exact:
            tree, weight, subsets = algorithm(G, terminals)
            print(f'Subsets checked: {subsets}')
        else:
            tree, weight = algorithm(G, terminals)
        elapsed = time.time() - start

        steiner_points = [v for v in tree.nodes() if v not in terminals]
        print(f'Weight: {weight}')
        print(f'Steiner points used: {steiner_points}')
        print(f'Edges: {list(tree.edges(data=True))}')
        print(f'Time: {elapsed:.4f}s')
        print()


def compare_algorithms(algorithms, files, bf_max_nodes=25):
    results = []
    for filepath in files:
        G, terminals, name = parse_stp(filepath)
        row = {'instance': name, 'nodes': G.number_of_nodes(), 'terminals': len(terminals)}
        for alg_name, alg_func, is_exact in algorithms:
            if is_exact and G.number_of_nodes() > bf_max_nodes:
                row[alg_name] = None
                row[alg_name + '_time'] = None
                continue
            start = time.time()
            if is_exact:
                tree, weight, _ = alg_func(G, terminals)
            else:
                tree, weight = alg_func(G, terminals)
            elapsed = time.time() - start
            row[alg_name] = weight
            row[alg_name + '_time'] = elapsed
        results.append(row)

    alg_names = [name for name, _, _ in algorithms]
    header = f"{'Instance':<10} {'N':>4} {'T':>4} | " + "  ".join(f"{name[:6]:>6}" for name in alg_names)
    print(header)
    print('-' * len(header))
    bf_name = alg_names[0]
    for r in results:
        bf = r[bf_name]
        if bf is None:
            print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} |      -", end='')
        else:
            print(f"{r['instance']:<10} {r['nodes']:>4} {r['terminals']:>4} | {bf:>6.0f}", end='')
        for alg_name in alg_names[1:]:
            w = r[alg_name]
            if bf is not None and w == bf:
                print(f"  {'*':>6}", end='')
            elif bf is not None:
                diff = (w - bf) / bf * 100
                print(f"  {w:>3.0f}+{diff:.0f}%", end='')
            else:
                print(f"  {w:>6.0f}", end='')
        print()

    print()
    print("* = optimalno rešenje, - = preskočeno (preveliko za brute force)")
    return results


def plot_times(results, algorithms):
    alg_names = [name for name, _, _ in algorithms]
    instances = [r['instance'] for r in results]
    x = range(len(instances))
    width = 0.15
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, alg_name in enumerate(alg_names):
        times = []
        for r in results:
            t = r.get(alg_name + '_time')
            times.append(t if t is not None else 0)
        offset = (i - len(alg_names) / 2 + 0.5) * width
        bars = ax.bar([xi + offset for xi in x], times, width, label=alg_name, color=colors[i % len(colors)])

    ax.set_xlabel('Instanca')
    ax.set_ylabel('Vreme (s)')
    ax.set_title('Vreme izvršavanja po algoritmu')
    ax.set_xticks(list(x))
    ax.set_xticklabels(instances, rotation=45, ha='right')
    ax.legend()
    ax.set_yscale('log')
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_weights(results, algorithms):
    alg_names = [name for name, _, _ in algorithms]
    instances = [r['instance'] for r in results]
    x = range(len(instances))
    width = 0.15
    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, alg_name in enumerate(alg_names):
        weights = []
        for r in results:
            w = r.get(alg_name)
            weights.append(w if w is not None else 0)
        offset = (i - len(alg_names) / 2 + 0.5) * width
        ax.bar([xi + offset for xi in x], weights, width, label=alg_name, color=colors[i % len(colors)])

    ax.set_xlabel('Instanca')
    ax.set_ylabel('Težina Steinerovog stabla')
    ax.set_title('Poređenje težina po algoritmu')
    ax.set_xticks(list(x))
    ax.set_xticklabels(instances, rotation=45, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()